# Context rot & lost-in-the-middle

**Session 5 · small model (`llama3.2:3b`)**

More context is not free. Put one fact in a block of filler, ask for it back, and sweep the
filler size. Short context: the model finds the fact anywhere. Past a few thousand tokens the
small model only reliably recalls it when it sits at the **end** — the start and middle rot.
The demo only works if the filler is actually long, so we measure at 3 sizes.

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask, count_tokens, SMALL_MODEL


### Worked example

`recall_rate(size, position)` builds a context of ~`size` filler tokens with the fact at
`position`, asks 3 times, and returns how often the code came back. Sweep size × position.

In [ ]:
FACT = "The access badge code for room 214 is 8843."
Q = "What is the access badge code for room 214? Answer with the code only."
PARA = ("The facilities team reviewed the quarterly maintenance schedule on Monday. "
        "The north stairwell lighting will be upgraded to LED units next month. "
        "Visitor parking has moved to level 2 while resurfacing work continues. "
        "The coffee machine on floor 3 is due for its annual service. "
        "Recycling collection now happens on Tuesdays and Fridays. ")

def filler(n_tokens):
    s = ""
    while count_tokens(s) < n_tokens:
        s += PARA
    return s

def context(size, position):
    f = filler(size)
    half = len(f) // 2
    return {
        "start":  FACT + " " + f,
        "middle": f[:half] + " " + FACT + " " + f[half:],
        "end":    f + " " + FACT,
    }[position]

def recall_rate(size, position, tries=3):
    hits = 0
    for _ in range(tries):
        out = ask(f"Use only the following context to answer.\n\n{context(size, position)}\n\n{Q}",
                  model=SMALL_MODEL)
        hits += "8843" in out
    return hits, tries

for size in (400, 1500, 4000):
    row = "   ".join(f"{pos}: {h}/{t}" for pos in ("start", "middle", "end")
                      for h, t in [recall_rate(size, pos)])
    print(f"~{size:>4} filler tokens   {row}")


## Your turn - vary the example

1. Add a `6000` row. Does `end` finally break too?
2. Put TWO facts in (two rooms, two codes) at start and end, ask for both. Which is recalled?
3. Keep size at 4000 and `position="end"`, but append 500 tokens of *noise after* the fact.
   How much trailing noise buries an otherwise well-placed fact?
4. Practical takeaway: given this table, where in a RAG prompt should the retrieved chunk go,
   and how much filler is safe?


In [ ]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
